DecisionLens AI

This notebook trains and evaluates the first machine learning model for stance classification.

The following steps are performed:

- Load the processed datasets.
- Split features and target.
- Vectorize the text using TF-IDF.
- Train a Logistic Regression classifier.
- Evaluate the model.
- Save the trained model.

Import Libraries

In [38]:
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

Load Processed Datasets

In [39]:
train_df = pd.read_csv('/content/drive/MyDrive/PROYECT_5/train_processed.csv')
test_df = pd.read_csv('/content/drive/MyDrive/PROYECT_5/test_processed.csv')
validation_df = pd.read_csv('/content/drive/MyDrive/PROYECT_5/validation_processed.csv')

In [40]:
print(train_df.shape)
print(test_df.shape)
print(validation_df.shape)

(20974, 2)
(6315, 2)
(3208, 2)


In [41]:
train_df.head()

,argument,stance_WA
0,"""marriage"" isn't keeping up with the times. ab...",1
1,.a multi-party system would be too confusing a...,-1
2,`people reach their limit when it comes to the...,-1
3,"100% agree, should they do that, it would be a...",1
4,a ban on naturopathy creates a cohesive front ...,1


Split Features and Target

In [42]:
X_train = train_df['argument']
y_train = train_df['stance_WA']

X_test = test_df['argument']
y_test = test_df['stance_WA']

X_validation = validation_df['argument']
y_validation = validation_df['stance_WA']

TF-IDF Vectorization

In [43]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

In [44]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
X_validation_tfidf = vectorizer.transform(X_validation)

In [45]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)
print(X_validation_tfidf.shape)

(20974, 5000)
(6315, 5000)
(3208, 5000)


Train Model - Logistic Regression

In [46]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [48]:
print(len(vectorizer.vocabulary_))
print(vectorizer.idf_.shape)

5000
(5000,)


Model Evaluation

In [49]:
y_validation_pred = model.predict(X_validation_tfidf)
y_test_pred = model.predict(X_test_tfidf)

y_val_acc = accuracy_score(y_validation, y_validation_pred)
y_test_acc = accuracy_score(y_test, y_test_pred)
print(f'Validation Accuracy: {y_val_acc}')
print(f'Test Accuracy: {y_test_acc}')

Validation Accuracy: 0.649002493765586
Test Accuracy: 0.6799683293745051


In [50]:
y_val_confusion_mx = confusion_matrix(y_validation, y_validation_pred)
print(y_val_confusion_mx)

y_test_confusion_mx = confusion_matrix(y_test, y_test_pred)
print(y_test_confusion_mx)

[[1029  557]
 [ 569 1053]]
[[2291  846]
 [1175 2003]]


In [51]:
y_val_class_rep = classification_report(y_validation, y_validation_pred)
print(y_val_class_rep)

y_test_class_rep = classification_report(y_test, y_test_pred)
print(y_test_class_rep)

              precision    recall  f1-score   support

          -1       0.64      0.65      0.65      1586
           1       0.65      0.65      0.65      1622

    accuracy                           0.65      3208
   macro avg       0.65      0.65      0.65      3208
weighted avg       0.65      0.65      0.65      3208

              precision    recall  f1-score   support

          -1       0.66      0.73      0.69      3137
           1       0.70      0.63      0.66      3178

    accuracy                           0.68      6315
   macro avg       0.68      0.68      0.68      6315
weighted avg       0.68      0.68      0.68      6315



Save Model

In [52]:
joblib.dump(model, '/content/drive/MyDrive/PROYECT_5/logistic_regression_model.pkl')
joblib.dump(vectorizer, '/content/drive/MyDrive/PROYECT_5/tfidf_vectorizer.pkl')

['/content/drive/MyDrive/PROYECT_5/tfidf_vectorizer.pkl']

Compare Different Models

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'SVM': SVC(),
    'Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC()
}

In [ ]:
for name, model in models.items():
  model.fit(X_train_tfidf, y_train)
  y_val_pred = model.predict(X_validation_tfidf)
  y_test_pred = model.predict(X_test_tfidf)
  y_val_acc = accuracy_score(y_validation, y_val_pred)
  y_test_acc = accuracy_score(y_test, y_test_pred)
  y_val_confusion_mx = confusion_matrix(y_validation, y_val_pred)
  y_test_confusion_mx = confusion_matrix(y_test, y_test_pred)
  y_val_class_rep = classification_report(y_validation, y_val_pred)
  y_test_class_rep = classification_report(y_test, y_test_pred)

  print(f'{name} - Validation Accuracy: {y_val_acc}')
  print(f'{name} - Test Accuracy: {y_test_acc}')
  print(f'{name} - Validation Confusion Matrix:')
  print(y_val_confusion_mx)
  print(f'{name} - Test Confusion Matrix:')
  print(y_test_confusion_mx)
  print(f'{name} - Validation Classification Report:')
  print(y_val_class_rep)
  print(f'{name} - Test Classification Report:')
  print(y_test_class_rep)
  print('--------------------------------------------------------------------------------')

Decision Tree - Validation Accuracy: 0.6265586034912718
Decision Tree - Test Accuracy: 0.6199524940617577
Decision Tree - Validation Confusion Matrix:
[[ 978  608]
 [ 590 1032]]
Decision Tree - Test Confusion Matrix:
[[1967 1170]
 [1230 1948]]
Decision Tree - Validation Classification Report:
              precision    recall  f1-score   support

          -1       0.62      0.62      0.62      1586
           1       0.63      0.64      0.63      1622

    accuracy                           0.63      3208
   macro avg       0.63      0.63      0.63      3208
weighted avg       0.63      0.63      0.63      3208

Decision Tree - Test Classification Report:
              precision    recall  f1-score   support

          -1       0.62      0.63      0.62      3137
           1       0.62      0.61      0.62      3178

    accuracy                           0.62      6315
   macro avg       0.62      0.62      0.62      6315
weighted avg       0.62      0.62      0.62      6315

--------

Final Model Selection

Several lightweight machine learning models were evaluated using the same TF-IDF features.

Although Random Forest achieved the highest accuracy, the improvement over Logistic Regression was relatively small (approximately 1%).

Logistic Regression was selected as the final model because it provides competitive performance while being simpler, faster, and more efficient for deployment. It has low memory requirements, fast prediction time, and integrates well with the planned Streamlit web application.

For these reasons, Logistic Regression was chosen as the production model for DecisionLens AI.